In [1]:
from google.colab import auth
from google.colab import drive
from google.cloud import bigquery

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
auth.authenticate_user()
drive.mount('/content/drive')

Mounted at /content/drive



# Configuration

In [ ]:
# Load Datset
file_path = '/content/drive/MyDrive/Colab Notebooks/LendingClub/archive/Loan_status_2007-2020Q3.csv'
df = pd.read_csv(file_path, dtype = str, low_memory=False)

# BigQuery Project ID
PROJECT_ID = '' # Your Project ID

DATASET_ID = 'lendingclub'

TABLE_ID = 'loans_2007_2020'


# Clean Dataset

In [4]:

initial_rows = len(df)

# Remove embedded heaer rows (loan_amnt = "loan_amnt")
if 'loan_amnt' in df.columns:
  df = df[df['loan_amnt'].astype(str) != 'loan_amnt']

# Remove completely empty rows
df = df.dropna(how = 'all')

# Reset Index
df = df.reset_index(drop = True)


# Convert Data Types

In [5]:
# FIX PERCENTAGE COLUMNS
percentage_columns = [
    'int_rate', 'revol_util', 'bc_util', 'all_util', 'il_util',
    'percent_bc_gt_75', 'dti', 'dti_joint', 'sec_app_revol_util',
    'pct_tl_nvr_dlq', 'settlement_percentage',
]

for col in percentage_columns:
    if col in df.columns:
      # Remove spaces and %, then convert to numeric
      df[col] = df[col].astype(str).str.strip().str.replace('%', '', regex= False)
      df[col] = pd.to_numeric(df[col], errors = 'coerce')

      # Also create decimal version if needed for calculation later
      df[f'{col}_decimal'] = df[col] / 100

In [ ]:
# Key numeric columns to convert
numeric_float = [
    'loan_amnt', 'funded_amnt', 'int_rate', 'installment',
    'annual_inc', 'dti', 'revol_bal', 'revol_util',
    'total_pymnt', 'total_rec_prncp', 'total_rec_int',
    'bc_util', 'all_util', 'tot_hi_cred_lim', 'tot_cur_bal',
    'total_bal_ex_mort', 'avg_cur_bal',
]

numeric_int = [
    'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc',
    'acc_open_past_24mths', 'mort_acc', 'pub_rec_bankruptcies',
    'num_actv_bc_tl', 'num_bc_tl', 'num_tl_30dpd',
]


# Convert float columns
converted_float = 0
for col in numeric_float:
  if col in df.columns:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    converted_float += 1

# Convert integer columns (use nullable Int64)
converted_int = 0
for col in numeric_int:
  if col in df.columns:
    df[col] = pd.to_numeric(df[col], errors = 'coerce').astype('Int64')
    converted_int += 1

print(f"Converted {converted_float} float columns")
print(f"Converted {converted_int} integer columns")

# Check initial_list_status
if 'initial_list_status' in df.columns:
    print(df['initial_list_status'].value_counts())
    print("   (These are strings 'w' and 'f', NOT booleans!)")


✅ Converted 17 float columns
✅ Converted 13 integer columns

📊 initial_list_status values:
initial_list_status
w    2139434
f     786058
Name: count, dtype: int64
   (These are strings 'w' and 'f', NOT booleans!)


# Final Validation

In [ ]:
rows_before = len(df)

# Only keep rows with valid loan amounts and interest rates
df = df[
    df['loan_amnt'].notna() &
    df['int_rate'].notna() &
    (df['loan_amnt'] > 0) &
    (df['int_rate'] > 0)
]

rows_removed = rows_before - len(df)


if len(df) == 0:
    print("ERROR: No valid data remaining!")
    print("Check your CSV file - it may be corrupted.")
    raise ValueError("No valid data")


   Removed: 1 rows with invalid loan_amnt or int_rate

✅ Final dataset: 2,925,492 rows


In [ ]:
# Show sample
print("Sample of cleaned data:")
sample_cols = ['loan_amnt', 'int_rate', 'grade', 'loan_status', 'revol_util']
print(df[sample_cols].head(10).to_string())


📊 Sample of cleaned data:
   loan_amnt  int_rate grade  loan_status  revol_util
0     5000.0     10.65     B   Fully Paid        83.7
1     2500.0     15.27     C  Charged Off         9.4
2     2400.0     15.96     C   Fully Paid        98.5
3    10000.0     13.49     C   Fully Paid        21.0
4     3000.0     12.69     B   Fully Paid        53.9
5     5000.0      7.90     A   Fully Paid        28.3
6     7000.0     15.96     C   Fully Paid        85.6
7     3000.0     18.64     E   Fully Paid        87.5
8     5600.0     21.28     F  Charged Off        32.6
9     5375.0     12.69     B  Charged Off        36.5


In [9]:
# Save the cleaned dataset to csv
cleaned_path = '/content/drive/MyDrive/Colab Notebooks/LendingClub/archive/Loan_status_2007-2020Q3_cleaned.csv'
df.to_csv(cleaned_path,  index=False)

# CREATE BIGQUERY DATASET

In [10]:
client = bigquery.Client(project=PROJECT_ID)

dataset_ref = f'{PROJECT_ID}.{DATASET_ID}'
dataset = bigquery.Dataset(dataset_ref)
dataset.location = 'US'

try:
  client.create_dataset(dataset, exists_ok=True)
  print(f"Dataset '{DATASET_ID}' is ready")
except Exception as e:
  print(f"Error creating dataset: {e}")

Dataset 'lendingclub' is ready


# UPLOAD TO BIGQUERY

In [11]:
full_table_id = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}'

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

try:
    # Upload
    job = client.load_table_from_dataframe(
        df,
        full_table_id,
        job_config=job_config
    )

    # Wait for completion
    print("\nUploading", end="", flush=True)
    import time
    while job.state != 'DONE':
        print(".", end="", flush=True)
        time.sleep(3)
        job.reload()

    if job.errors:
        print(f"Errors: {job.errors}")
    else:
        print("Upload successful!")

except Exception as e:
    print(f"Upload failed: {e}")
    raise


📤 Uploading to: lendingclub-488900.lendingclub.loans_2007_2020
   Rows: 2,925,492

Uploading.......... Done!

✅ Upload successful!


# VERIFY

In [12]:
table = client.get_table(full_table_id)
print(f"TABLE INFORMATION:")
print(f"   Rows: {table.num_rows:,}")
print(f"   Columns: {len(table.schema)}")
print(f"   Size: {table.num_bytes / 1024**2:.1f} MB")


📊 TABLE INFORMATION:
   Rows: 2,925,492
   Columns: 152
   Size: 2254.8 MB
